<h1>THE <em>do</em>WARC<em>lite</em> NOTEBOOK</h1>

<h2>CRAWLING AND REPLAYING WARC FILES</H2>

<h4>Install all dependencies locally via terminal before installing them in the notebook, to prevent issues with permissions.</h4>

In [ ]:
!pip install bash_kernel

In [ ]:
!pip install docker

In [ ]:
!pip install pywb

<h4>Pull the <code>Broswertrix Crawler</code> dockerised image.</h4>

In [ ]:
!docker pull webrecorder/browsertrix-crawler

<h4>Crawl the targeted website and write a WARC file.</H4>

In [ ]:
!docker run -v $PWD/crawls:/crawls/ -it webrecorder/browsertrix-crawler crawl --url https://yourwebsite.org/ --generateWARC --text --collection yourwebsite_collection

<h4>Create a collection for the archived website to replay in the browser.</h4>

In [ ]:
!wb-manager init yourwebsite_collection

In [ ]:
!wb-manager add yourwebsite_collection path/to/your/warc/file/yourwebsite.org.warc.gz

<h4>Launch the <code>pywb</code> server to replay the archived website.</h4>

In [ ]:
!wayback

Now you can access your <code>Pywb Wayback Machine</code> instance and your WARC collection in a new browser window with your local host port URL (e.g., <code>http://localhost:8080</code>).

<h2>CREATING AN <CODE>SQLITE</CODE> DATABASE SCHEMA USING THE <EM>doWARClite</EM> ONTOLOGY</H2>

<h4>Download the <code><em>do</em>WARClite</code> ontology from the <code>DOWARC<code> GitHub repository:</h4>

(https://github.com/DOWARC/dowarc/blob/main/dowarcLITE.owx)

<h4>Stream the <code><em>do</em>WARClite</code> ontology

In [ ]:
from rdflib import Graph
from rdflib.namespace import RDFS, OWL

# Load the RDF file
g = Graph()
g.parse("your/local/path/to/dowarcLITE.rdf", format="xml")

# Query for all classes
query = """
SELECT DISTINCT ?class WHERE {
    ?class a owl:Class .
}
"""
results = g.query(query)

# Print classes - fix: use row[0] or str(row.class)
print("=== ALL CLASSES ===")
for row in results:
    print(row[0])  # Access by index
    # OR: print(row.class) - but this doesn't work because 'class' is a keyword

# Get class hierarchy
query_hierarchy = """
SELECT ?subclass ?superclass WHERE {
    ?subclass rdfs:subClassOf ?superclass .
}
"""
hierarchy = g.query(query_hierarchy)

print("\n=== CLASS HIERARCHY ===")
for row in hierarchy:
    subclass = row[0]  # First column
    superclass = row[1]  # Second column
    print(f"{subclass} -> {superclass}")

<h4>Create the SQLite database schema using the <code><em>do</em>WARClite</code> ontology.</h4>

Set up all necessary variables <em>before</em> creating the database (e.g., the database name for <code>{db_name}</code>.

In [ ]:
import os
import sqlite3

db_name = "the-name-of-your-database"

print(f"Creating the {db_name} SQLite database file")

conn = sqlite3.connect(db_name)
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")

cursor.execute('''
    CREATE TABLE IF NOT EXISTS dctermsPublisher (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        publisher_name TEXT UNIQUE NOT NULL
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS premisStorageMedium (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        carrierID TEXT UNIQUE NOT NULL
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS premisStorageLocation (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        carrierLocation TEXT UNIQUE NOT NULL
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS ArchivedWebResource (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        URLvalue TEXT UNIQUE NOT NULL,
        publisher_name TEXT,

        FOREIGN KEY (publisher_name) REFERENCES dctermsPublisher(publisher_name)
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS WarcCollection (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        warc_collection_id TEXT UNIQUE NOT NULL,
        URLvalue TEXT,
        carrierID TEXT,
        carrierLocation TEXT,

        FOREIGN KEY (URLvalue) REFERENCES ArchivedWebResource(URLvalue),
        FOREIGN KEY (carrierID) REFERENCES premisStorageMedium(carrierID),
        FOREIGN KEY (carrierLocation) REFERENCES premisStorageLocation(carrierLocation)
    )
''')


cursor.execute('''
    CREATE TABLE IF NOT EXISTS ConfigFile (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        crawl_order_ID TEXT UNIQUE NOT NULL,
        URLvalue TEXT,
        carrierID TEXT,
        carrierLocation TEXT,
        size REAL,

        FOREIGN KEY (URLvalue) REFERENCES ArchivedWebResource(URLvalue),
        FOREIGN KEY (carrierID) REFERENCES premisStorageMedium(carrierID),
        FOREIGN KEY (carrierLocation) REFERENCES premisStorageLocation(carrierLocation)
    )
''')


cursor.execute('''
    CREATE TABLE IF NOT EXISTS Crawling (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        crawl_ID TEXT UNIQUE NOT NULL,
        URLvalue TEXT,
        sourceAccessedOn TEXT,
        crawl_order_ID TEXT,

        FOREIGN KEY (crawl_order_ID) REFERENCES ConfigFile(crawl_order_ID),
        FOREIGN KEY (URLvalue) REFERENCES ArchivedWebResource(URLvalue)
    )
''')

crawl_order_ID
cursor.execute('''
    CREATE TABLE IF NOT EXISTS WarcFile (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        warc_file_name TEXT UNIQUE NOT NULL,
        URLvalue TEXT,
        crawl_ID TEXT,
        carrierID TEXT,
        carrierLocation TEXT,
        warc_collection_id TEXT,
        crawl_order_ID TEXT,
        WarcFileSignature TEXT,
        generatedAtTime TEXT,
        size REAL,
    
        FOREIGN KEY (URLvalue) REFERENCES ArchivedWebResource(URLvalue),
        FOREIGN KEY (crawl_ID) REFERENCES Crawling(crawl_ID),
        FOREIGN KEY (carrierID) REFERENCES premisStorageMedium(carrierID),
        FOREIGN KEY (carrierLocation) REFERENCES premisStorageLocation(carrierLocation),
        FOREIGN KEY (warc_collection_id) REFERENCES WarcCollection(warc_collection_id),
        FOREIGN KEY (crawl_order_ID) REFERENCES ConfigFile(crawl_order_ID)
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS premisSignature (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        WarcFileSignature TEXT UNIQUE NOT NULL,
        warc_file_name TEXT,

        FOREIGN KEY (warc_file_name) REFERENCES WarcFile(warc_file_name)    

    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS Indexing (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        index_timestamp TEXT UNIQUE NOT NULL,
        warc_file_name TEXT,

        FOREIGN KEY (warc_file_name) REFERENCES WarcFile(warc_file_name)
    )
''')


cursor.execute('''
    CREATE TABLE IF NOT EXISTS WarcCdxFile (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        cdx_file_name TEXT UNIQUE NOT NULL,
        warc_file_name TEXT,
        carrierID TEXT,
        carrierLocation TEXT,
        size REAL,
        index_timestamp TEXT,

        FOREIGN KEY (warc_file_name) REFERENCES WarcFile(warc_file_name),
        FOREIGN KEY (carrierID) REFERENCES premisStorageMedium(carrierID),
        FOREIGN KEY (carrierLocation) REFERENCES premisStorageLocation(carrierLocation),
        FOREIGN KEY (index_timestamp) REFERENCES Indexing(index_timestamp)
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS Extracting (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        extract_timestamp TEXT UNIQUE NOT NULL,
        warc_file_name TEXT,
        sourceAccessedOn TEXT,

        FOREIGN KEY (warc_file_name) REFERENCES WarcFile(warc_file_name)
    )
''')


cursor.execute('''
    CREATE TABLE IF NOT EXISTS WarcRecord (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        WARC_Record_ID TEXT UNIQUE NOT NULL,
        extract_timestamp TEXT,
        warc_file_name TEXT,
        WARC_Date TEXT,
        WARC_Type TEXT,
        WARC_Truncated TEXT,
        WARC_Profile TEXT,
        WARC_Segment_Origin_ID TEXT,
        WARC_Segment_Number INTEGER,
        WARC_Segment_Total_Length INTEGER,
        WARC_Warcinfo_ID TEXT,
        URLvalue TEXT,
        WARC_Filename TEXT,
        qa_timestamp TEXT,
        WARC_Concurrent_To TEXT,
        WARC_Refers_To TEXT,
        offset INTEGER,


        FOREIGN KEY (warc_file_name) REFERENCES WarcFile(warc_file_name),
        FOREIGN KEY (extract_timestamp) REFERENCES Extracting(extract_timestamp),
        FOREIGN KEY (URLvalue) REFERENCES ArchivedWebResource(URLvalue)
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS QA (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        qa_timestamp TEXT UNIQUE NOT NULL,
        WARC_Record_ID TEXT,
        warc_file_name TEXT,
        cdx_file_name TEXT,
        note TEXT,
        sourceAccessedOn TEXT,

        FOREIGN KEY (WARC_Record_ID) REFERENCES WarcRecord(WARC_Record_ID),
        FOREIGN KEY (warc_file_name) REFERENCES WarcFile(warc_file_name),
        FOREIGN KEY (cdx_file_name) REFERENCES WarcCdxFile(cdx_file_name)
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS WarcRecordContentBlock (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        WARC_Block_Digest TEXT UNIQUE NOT NULL,
        WARC_Record_ID TEXT,
        content_type TEXT,
        extract_timestamp TEXT,
        qa_timestamp TEXT,

        FOREIGN KEY (WARC_Record_ID) REFERENCES WarcRecord(WARC_Record_ID),
        FOREIGN KEY (extract_timestamp) REFERENCES Extracting(extract_timestamp),
        FOREIGN KEY (qa_timestamp) REFERENCES QA(qa_timestamp)
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS WarcRecordHeader (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        HeaderValue TEXT UNIQUE NOT NULL,
        WARC_Record_ID TEXT,
        extract_timestamp TEXT,
        qa_timestamp TEXT,
        
        FOREIGN KEY (WARC_Record_ID) REFERENCES WarcRecord(WARC_Record_ID),
        FOREIGN KEY (extract_timestamp) REFERENCES Extracting(extract_timestamp),
        FOREIGN KEY (qa_timestamp) REFERENCES QA(qa_timestamp)
    )
''')

cursor.execute('''
    CREATE TABLE WarcRecordPayload (
        WARC_Payload_Digest TEXT NOT NULL,
        WARC_Record_ID TEXT NOT NULL,
        PayloadValue BLOB,
        extract_timestamp TEXT,
        qa_timestamp TEXT,
        WARC_Identified_Payload_Type TEXT,
        
        PRIMARY KEY (WARC_Payload_Digest, WARC_Record_ID),
        FOREIGN KEY (WARC_Record_ID) REFERENCES WarcRecord(WARC_Record_ID),
        FOREIGN KEY (extract_timestamp) REFERENCES Extracting(extract_timestamp),
        FOREIGN KEY (qa_timestamp) REFERENCES QA(qa_timestamp)
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS WarcGraph (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        graphName TEXT UNIQUE NOT NULL,
        URLvalue TEXT,
        warc_file_name TEXT,
        carrierID TEXT,
        carrierLocation TEXT,
    
        FOREIGN KEY (URLvalue) REFERENCES ArchivedWebResource(URLvalue),
        FOREIGN KEY (warc_file_name) REFERENCES WarcFile(warc_file_name),
        FOREIGN KEY (carrierID) REFERENCES premisStorageMedium(carrierID),
        FOREIGN KEY (carrierLocation) REFERENCES premisStorageLocation(carrierLocation)
    )
''')

conn.commit(

)
conn.close()

<H4>Stream the database schema and data-types information</H4>

In [ ]:
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# Get all tables and their schemas
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("=== DATABASE SCHEMA ===\n")
for table in tables:
    table_name = table[0]
    print(f"Table: {table_name}")
    
    # Get schema for this table
    cursor.execute(f"SELECT sql FROM sqlite_master WHERE type='table' AND name='{table_name}'")
    schema = cursor.fetchone()[0]
    print(schema)
    
    # Get column info
    cursor.execute(f"PRAGMA table_info({table_name})")
    columns = cursor.fetchall()
    print("Columns:")
    for col in columns:
        print(f"  - {col[1]} ({col[2]})")

conn.close()

<h2>WRITING <code>WARC</code> DATA TO THE SQLite DATABASE FILE.</H2>

<h4>Install the <code>CDXJ Indexer</code> to index the <code>WARC</code> file.

In [ ]:
!pip install cdxj-indexer

<h4>Index the <code>WARC</code> file and create the <code>cdxj</code> file.</h4>

Import <code>datetime</code> to create a timestamp for the indexing process. Ensure variables are carried over from previous cell(s), or created in this new one. (e.g., <code>{warc_filename}</code>.)

In [ ]:
import subprocess
from datetime import datetime

warc_filename = "yourwebsite.org.warc.gz"
cdxj_file = f"{warc_filename}.cdxj"

indexing_timestamp = datetime.now().isoformat()
indexing_timestamp_str = str(indexing_timestamp)
clean_indexing_timestamp = indexing_timestamp_str.replace(' ', '_')

# Initialize size counter
cdxj_file_size = 0

# Run the indexer
result = subprocess.run(
    ['cdxj-indexer', '-11', warc_file],
    capture_output=True, text=True
)

# Save output to file
with open(cdxj_file, 'w') as f:
    f.write(result.stdout)

with open(cdxj_file, 'rb') as f:
    for chunk in iter(lambda: f.read(8192), b''):
        cdxj_file_size += len(chunk)

<h4>Stream the first rows of the <code>cdxj</code> file to inspect the data.</h4>

In [ ]:
print("=== CDXJ FILE PREVIEW (first 10 lines) ===\n")
with open(cdxj_file, 'r') as f:
    for i, line in enumerate(f):
        if i >= 10:  # Show first 10 lines
            break
        columns = line.strip().split()
        if i == 0:
            print(f"HEADERS ({len(columns)} columns): {columns}\n")
        else:
            print(f"Row {i}: {columns[:5]}...")  # Show first 5 columns only

<h4>Read the <code>cdxj</code> file in a <code>pandas</code> data frame.</h4>

The <code>cdxj</code> documentation explains the meaning of the headers and was used to define the <code>data frame</code> headers.

(https://iipc.github.io/warc-specifications/specifications/cdx-format/cdx-2015/)

In [ ]:
import pandas as pd

cdxj_df = pd.read_csv(cdxj_file, sep=" ", index_col=None, header=0)
cdxj_df.drop(columns='V', inplace=True)
cdxj_df.drop(columns='g', inplace=True)

cdxj_df.columns=["surt", "date", "original url", "mime type of original document", "response code", "new style checksum", "redirect", "meta tags (AIF)", "compressed record size", "compressed arc file offset", "file name"]

cdxj_cols = cdxj_df.columns.tolist()

print(f"The available headers are: {cdxj_cols}")

print(cdxj_df.head())

<h4>Install <code>WARCIO</code> and <code>FastWARC</code>to extract data from the <code>WARC</code> file.</h4>

In [ ]:
!pip install warcio

In [ ]:
!pip install fastwarc

<h4>Use <code>WARCIO</code> to extract the following <code>WARC</code> file elements and write them to a <code>json</code> file:</h4>

In [ ]:
!warcio index -f offset,length,filename,content-type,http:content-type,warc-target-uri /path/to/yourwebsite.org.warc.gz >yourwebsite-WARCIO.json

<h4>Read the <code>WARCIO</code> <code>json</code> in a data frame:</h4>

In [ ]:
warcio_df = pd.read_json('yourwebsite-WARCIO.json', lines=True)

warcio_cols = warcio_df.columns.tolist()

print(f"The WARCIO headers are: {warcio_cols}")

print(warcio_df.head())

In [ ]:
print(warcio_df.columns.tolist())

<h4>Store a subset of the <code>WARCIO</code>data frame for later reuse:</h4>

In [ ]:
df = pd.read_json('yourwebsite-WARCIO.json', lines=True)

warcio_json_df = df[['offset', 'filename', 'warc-target-uri']]
print(warcio_json_df.head())

<h4>Read the crawl <code>*.log</code> file generated by Browsertrix.</h4>

In [ ]:
crawl_log_file_name = '/path/to/your/crawl/log/file.log'

with open(crawl_log_file_name, 'rb') as f:
    first_100_bytes = f.read(100)
    print(first_100_bytes)
    print(repr(first_100_bytes))

<h4>Store the crawl <code>*.log</code> file in a <code>pandas</code> data frame to:
<ol>
    <li>automatically extract the seed URL</li>
    <li>extract the crawl timestamp</li>
    <li>create and store an identifier for the crawl (if it is not already been provided by a pre-existing crawl configuration file).</li>
</o1></h4>

In [ ]:
crawl_log_df = pd.read_json(crawl_log_file_name, lines=True)
row_1 = crawl_log_df.iloc[1]
print(row_1)

In [ ]:
seed_URL = row_1['details'][0]['url']
crawl_timestamp = row_1['timestamp']
crawl_timestamp_str = str(crawl_timestamp)
clean_crawl_timestamp = crawl_timestamp_str.replace(' ', '_')
crawl_ID = f"{clean_crawl_timestamp}_{seed_URL}"
print(f"Seed URL: {seed_URL}")
print(f"WARC file generatedAtTime: {clean_crawl_timestamp}")
print(f"Crawl ID is {crawl_ID}")

<h4>Write data in the <code>ArchivedWebResource</code> table</h4>.

Check that the data has been written to the table.

In [ ]:
import sqlite3

warc_target_uri = warcio_df['warc-target-uri']

conn = sqlite3.connect(db_name)
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")

cursor.execute('''
    INSERT OR REPLACE INTO ArchivedWebResource 
    (URLvalue)
    VALUES (?)
''', (
    seed_URL,
))

for warc_target_uri in warcio_df['warc-target-uri']:
    if warc_target_uri is not None and pd.notna(warc_target_uri):  # Skip NULL/NaN
        cursor.execute('''
            INSERT OR REPLACE INTO ArchivedWebResource 
            (URLvalue)
            VALUES (?)
        ''', (
            warc_target_uri,
        ))


conn.commit()
print("URLs saved to ArchivedWebResource table")

In [ ]:
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# Get column names and values to check
cursor.execute("PRAGMA table_info(ArchivedWebResource)")
columns = [col[1] for col in cursor.fetchall()]
print(f"Columns: {columns}")

cursor.execute("SELECT * FROM ArchivedWebResource")
rows = cursor.fetchall()

print(f"\nData ({len(rows)} rows):")
for row in rows:
    print(row)

conn.close()

<h4>Write data in the <code>Crawling</code> table</h4>.

Check that the data has been written to the table.

In [ ]:
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")
cursor.execute('''
    INSERT OR REPLACE INTO Crawling 
    (crawl_ID, URLvalue, sourceAccessedOn)
    VALUES (?, ?, ?)
''', (
    crawl_ID, seed_URL, clean_crawl_timestamp
))

conn.commit()

In [ ]:
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# Get column names and values to check
cursor.execute("PRAGMA table_info(Crawling)")
columns = [col[1] for col in cursor.fetchall()]
print(f"Columns: {columns}")

cursor.execute("SELECT * FROM Crawling")
rows = cursor.fetchall()

print(f"\nData ({len(rows)} rows):")
for row in rows:
    print(row)

conn.close()

<h4>Write data in the <code>WarcFile</code> and <code>premisSignature</code> tables.</h4>

Check that the data has been written to the table.

In [ ]:
import hashlib

conn = sqlite3.connect(db_name)
cursor = conn.cursor()

sha256_hash = hashlib.sha256()
warc_file_size = 0

with open(warc_full_path, 'rb') as f:
    for chunk in iter(lambda: f.read(8192), b''):
        sha256_hash.update(chunk)
        warc_file_size += len(chunk)

warc_file_hash = sha256_hash.hexdigest()

print(f"WARC file size: {warc_file_size:,}")
print(f"SHA256 hash: {warc_file_hash}")
print(f"Seed URL: {seed_URL}")
print(f"WARC file generatedAtTime: {clean_crawl_timestamp}")

cursor.execute("PRAGMA foreign_keys = ON;")

cursor.execute('''
    INSERT OR REPLACE INTO WarcFile 
    (warc_file_name, WarcFileSignature, size, generatedAtTime, URLvalue, crawl_ID)
    VALUES (?, ?, ?, ?, ?, ?)
''', (
    warc_filename, warc_file_hash, warc_file_size, clean_crawl_timestamp, seed_URL, crawl_ID
))

cursor.execute('''
    INSERT OR REPLACE INTO premisSignature
    (warc_file_name, WarcFileSignature)
    VALUES(?, ?)
''', (
    warc_filename, warc_file_hash
))

conn.commit()
conn.close()

print(f"Hash and size saved for {warc_filename}")


In [ ]:
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# Get column names and values to check
cursor.execute("PRAGMA table_info(WarcFile)")
columns = [col[1] for col in cursor.fetchall()]
print(f"Columns: {columns}")

cursor.execute("SELECT * FROM WarcFile")
rows = cursor.fetchall()

print(f"\nData ({len(rows)} rows):")
for row in rows:
    print(row)

conn.close()

In [ ]:
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# Get column names and values to check
cursor.execute("PRAGMA table_info(premisSignature)")
columns = [col[1] for col in cursor.fetchall()]
print(f"Columns: {columns}")

cursor.execute("SELECT * FROM premisSignature")
rows = cursor.fetchall()

print(f"\nData ({len(rows)} rows):")
for row in rows:
    print(row)

conn.close()

<h4>Write data in the <code>Indexing</code> table.</h4>

Check that the data has been written to the table.

In [ ]:
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")
cursor.execute('''
    INSERT OR REPLACE INTO Indexing 
    (index_timestamp, warc_file_name)
    VALUES (?, ?)
''', (clean_indexing_timestamp, warc_filename,
))

conn.commit()

In [ ]:
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# Get column names and values to check
cursor.execute("PRAGMA table_info(Indexing)")
columns = [col[1] for col in cursor.fetchall()]
print(f"Columns: {columns}")

cursor.execute("SELECT * FROM Indexing")
rows = cursor.fetchall()

print(f"\nData ({len(rows)} rows):")
for row in rows:
    print(row)

conn.close()

<h4>Write data in the <code>WarcCdxFile</code> table.</h4>

Check that the data has been written to the table.

In [ ]:
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")
cursor.execute('''
    INSERT OR REPLACE INTO WarcCdxFile 
    (cdx_file_name, index_timestamp, warc_file_name, size)
    VALUES (?, ?, ?, ?)
''', (
    cdxj_file, clean_indexing_timestamp, warc_filename, cdxj_file_size 
))

conn.commit()

In [ ]:
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# Get column names and values to check
cursor.execute("PRAGMA table_info(WarcCdxFile)")
columns = [col[1] for col in cursor.fetchall()]
print(f"Columns: {columns}")

cursor.execute("SELECT * FROM WarcCdxFile")
rows = cursor.fetchall()

print(f"\nData ({len(rows)} rows):")
for row in rows:
    print(row)

conn.close()

<h4>Extract data with WARCIO and write it to the <code>Extracting</code> and <code>WarcRecord</code> tables.</h4>

Check that the data has been written to the table.

In [ ]:
from warcio.archiveiterator import ArchiveIterator
import gzip
from datetime import datetime

extraction_timestamp = datetime.now().isoformat()
print(f"WARCIO extraction started at: {extraction_timestamp}")


conn = sqlite3.connect(db_name)
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")


cursor.execute('''
    INSERT OR REPLACE INTO Extracting 
    (extract_timestamp, warc_file_name)
    VALUES (?, ?)
''', (
    extraction_timestamp, warc_filename
))


with gzip.open(warc_full_path, 'rb') as stream:
    for record in ArchiveIterator(stream):
        warc_record_id = record.rec_headers.get_header('WARC-Record-ID')

        cursor.execute('''
            INSERT OR REPLACE INTO WarcRecord 
            (WARC_Record_ID, extract_timestamp, warc_file_name, WARC_Filename, WARC_Date, WARC_Type, WARC_Truncated, WARC_Profile, WARC_Segment_Origin_ID, WARC_Segment_Number, WARC_Segment_Total_Length, WARC_Warcinfo_ID, URLvalue)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            warc_record_id,
            extraction_timestamp,
            warc_filename,
            record.rec_headers.get_header('WARC-Filename'),
            record.rec_headers.get_header('WARC-Date'),
            record.rec_headers.get_header('WARC-Type'),
            record.rec_headers.get_header('WARC-Truncated'),
            record.rec_headers.get_header('WARC-Profile'),
            record.rec_headers.get_header('WARC-Segment-Origin-ID'),
            record.rec_headers.get_header('WARC-Segment-Number'),
            record.rec_headers.get_header('WARC-Segment-Total-Length'),
            record.rec_headers.get_header('WARC-Warcinfo-ID'),
            record.rec_headers.get_header('WARC-Target-URI')
        ))

print(f"Extraction timestamp: {extraction_timestamp}")


# Commit all changes to the database file
conn.commit()
conn.close()

In [ ]:
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# Get column names and values to check
cursor.execute("PRAGMA table_info(Extracting)")
columns = [col[1] for col in cursor.fetchall()]
print(f"Columns: {columns}")

cursor.execute("SELECT * FROM Extracting")
rows = cursor.fetchall()

print(f"\nData ({len(rows)} rows):")
for row in rows:
    print(row)

conn.close()

In [ ]:
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# Get column names and values to check
cursor.execute("PRAGMA table_info(WarcRecord)")
columns = [col[1] for col in cursor.fetchall()]
print(f"Columns: {columns}")

cursor.execute("SELECT * FROM WarcRecord")
rows = cursor.fetchall()

print(f"\nData ({len(rows)} rows):")
for row in rows:
    print(row)

conn.close()

<h4>Extract data with WARCIO and write it to the <code>WarcRecordContentBlock</code> table.</h4>

Check that the data has been written to the table.

In [ ]:
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# Generate timestamp for this extraction run
extraction_timestamp = datetime.now().isoformat()
print(f"Extraction timestamp: {extraction_timestamp}")

with open(warc_full_path, 'rb') as stream:
    for record in ArchiveIterator(stream):
        warc_block_digest = record.rec_headers.get_header('WARC-Block-Digest')
        warc_record_id = record.rec_headers.get_header('WARC-Record-ID')
        content_type = record.rec_headers.get_header('Content-Type')

        if warc_block_digest:
        
            cursor.execute('''
                INSERT INTO WarcRecordContentBlock 
                (WARC_Block_Digest, WARC_Record_ID, extract_timestamp, content_type)
                VALUES (?, ?, ?, ?)
            ''', (
                warc_block_digest,
                warc_record_id,
                extraction_timestamp,
                content_type
            ))
        
print(f"Named fields saved to WarcRecordContentBlock table")
print(f"Extraction timestamp: {extraction_timestamp}")


# Commit all changes to the database file
conn.commit()
conn.close()

In [ ]:
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# Get column names and values to check
cursor.execute("PRAGMA table_info(WarcRecordContentBlock)")
columns = [col[1] for col in cursor.fetchall()]
print(f"Columns: {columns}")

cursor.execute("SELECT * FROM WarcRecordContentBlock")
rows = cursor.fetchall()

print(f"\nData ({len(rows)} rows):")
for row in rows:
    print(row)

conn.close()

<h4>Extract data with FastWARC and write it to the <code>WarcRecordHeader</code> table.</h4>

Check that the data has been written to the table.

In [ ]:
# Connect to database
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# Generate timestamp for this extraction run
extraction_timestamp = datetime.now().isoformat()
print(f"Extraction timestamp: {extraction_timestamp}")

for index, row in warcio_json_df.iterrows():
    warc_rec_offset = row['offset']
    warc_filename = row['filename']
    
    # 1. Extract WARC-Record-ID from headers
    id_result = subprocess.run(
        ['fastwarc', 'extract', warc_full_path, str(warc_rec_offset), '--headers'],
        capture_output=True, text=True
    )

    # Parse WARC-Record-ID
    warc_record_id = None
    for line in id_result.stdout.split('\n'):
        if line.startswith('WARC-Record-ID:'):
            warc_record_id = line.split(':', 1)[1].strip()
            break

    # 2. Extract HEADERS 
    header_data = subprocess.run(
        ['fastwarc', 'extract', warc_full_path, str(warc_rec_offset), '--headers'],
        capture_output=True, text=False  # Keep as bytes
    )
      
        # Save WarcRecordHeader and WARC-Record-ID to WarcRecordHeader table with timestamp
    cursor.execute('''
        INSERT OR REPLACE INTO WarcRecordHeader 
        (WARC_Record_ID, HeaderValue, extract_timestamp)
        VALUES (?, ?, ?)
    ''', (
        warc_record_id,
        header_data.stdout,
        extraction_timestamp
    ))
        
    print(f"Header saved for offset {warc_rec_offset} (ID: {warc_record_id})")


conn.commit()
conn.close()

print(f"Extraction timestamp: {extraction_timestamp}")

<h4>Extract data with WARCIO and write it to the <code>WarcRecordPayload</code> table.</h4>

In [ ]:
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# Generate timestamp for this extraction run
extraction_timestamp = datetime.now().isoformat()
print(f"Extraction timestamp: {extraction_timestamp}")

with open(warc_full_path, 'rb') as stream:
    for record in ArchiveIterator(stream):
        warc_payload_digest = record.rec_headers.get_header('WARC-Payload-Digest')
        warc_record_id = record.rec_headers.get_header('WARC-Record-ID')

        if warc_payload_digest:
        
            cursor.execute('''
                INSERT OR REPLACE INTO WarcRecordPayload 
                (WARC_Payload_Digest, WARC_Record_ID)
                VALUES (?, ?)
            ''', (
                warc_payload_digest,
                warc_record_id
            ))
            
conn.commit()
conn.close()
print(f"Extraction timestamp: {extraction_timestamp}")

<h4>Extract data with FastWARC and write it to the <code>WarcRecordPayload</code> table.</h4>

Check that the data has been written to the table.

In [ ]:
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# Generate timestamp for this extraction run
extraction_timestamp = datetime.now().isoformat()
print(f"Extraction timestamp: {extraction_timestamp}")

for index, row in warcio_json_df.iterrows():
    warc_rec_offset = row['offset']
    
    # 1. Extract WARC-Record-ID from headers
    id_result = subprocess.run(
        ['fastwarc', 'extract', warc_full_path, str(warc_rec_offset), '--headers'],
        capture_output=True, text=True
    )
    
    
    # Parse WARC-Record-ID
    warc_record_id = None
    for line in id_result.stdout.split('\n'):
        if line.startswith('WARC-Record-ID:'):
            warc_record_id = line.split(':', 1)[1].strip()
            break

    # 2. Extract PAYLOAD and save to WarcRecordPayload table
    payload_result = subprocess.run(
        ['fastwarc', 'extract', warc_full_path, str(warc_rec_offset), '--payload'],
        capture_output=True, text=False  # Keep as bytes
    )

        # Save payload to WarcRecordPayload table with timestamp
        cursor.execute('''
            UPDATE WarcRecordPayload 
            SET PayloadValue = ?, extract_timestamp = ?
            WHERE WARC_Record_ID = ?
        ''', (
            payload_data, extraction_timestamp, warc_record_id
        ))
        
        print(f"Payload saved for offset {warc_rec_offset} (ID: {warc_record_id})")

conn.commit()
conn.close()

print(f"Rows updated: {cursor.rowcount}")
print(f"Extraction timestamp: {extraction_timestamp}")

<h2>Mapping the data to the <code><em>do</em>WARClite</code> ontology.</h2>

<h4>To create a mapping document compliant with the <code>RML</code> specifications, it is useful to use a tool like <code>YARRRML</code>. This will output a draft template for the mapping. Manual revision, however, is necessary and very resource-demanding. The file should be edited locally.</h4>

(https://rml.io/specs/rml/) and (https://rml.io/yarrrml/)

In [ ]:
!pip install owl2yarrrml

In [ ]:
!python3 -m owl2yarrrml -i dowarcLITE.rdf -o doWARCliteYARRRml-termsMapping.yml

<h3>Preview the <code>YARRRML</code> template in the notebook:</h4>

In [ ]:
with open('doWARCliteYARRRml-termsMapping.yml', 'r') as f:
    content = f.read()

print(content)

The YARRRML template output by <code>owl2yarrrml</code> might not be immediately reusable with a the RML engine of choice to represent the data in RDF. The template, however, is still useful.

<h2>Serializing the data as RDF triples using <code>Morph-KGC</code>.</h2>

<h4>To use <code>Morph-KGC</code> for the RDF serialization, the rule mapping document works best as an <code>RML</code> file. The one used to test the implementation of this notebook was created by manually re-writing the <code>YARRRML</code> file.</h4>

In [ ]:
!pip install morph-kgc[all]

In [ ]:
import morph_kgc
from pathlib import Path

RML_file = "/Path/to/your/RML/file/mapping_file.rml.ttl"

config = f"""
[CONFIGURATION]
output_format = N-TRIPLES
logging_level = INFO

[DataSource1]
mappings = {RML_file}
db_url = sqlite:///{db_name}
"""

graph = morph_kgc.materialize(config)
output_file = Path(wd_path) / db_name.replace('.db', '.ttl')
graph.serialize(destination=str(output_file), format="ttl")
print(f"Generated {len(graph)} triples -> {output_file}")

<h2>VISUALISING THE <code>WARC</code> RDF GRAPH.</h2>

<h4><CODE>PyVis</code> visualization of a graph subset.</h4>

In [ ]:
!pip install pyvis

In [ ]:
import rdflib
from rdflib import Graph

graph_file = output_file

g = rdflib.Graph()
g.parse(str(graph_file), format="turtle")
print(f"Loaded graph with {len(g)} triples")

# Print first 10 WarcRecord URIs from the triples-set
count = 0
for s in g.subjects(rdflib.RDF.type, rdflib.URIRef("https://github.com/DOWARC/dowarcLITE#WarcRecord")):
    print(f"  {s}")
    count += 1
    if count >= 10:
        break

After streaming the first WarcRecords URIs, we select one or two as targeted nodes of a small, subgraph to display in the notebook.

In [ ]:
g = rdflib.Graph()
g.parse(str(graph_file), format="turtle")
print(f"Loaded graph with {len(g)} triples")

# Targeted WarcRecord ID/URI node
target_uri = rdflib.URIRef("https://a-warc-id-from-the-list-output-above.com")

# Create subgraph around this node
subgraph = rdflib.Graph()

# Add outgoing triples taking this node as subject, with 1 outgoing hop
for s, p, o in g.triples((target_uri, None, None)):
    subgraph.add((s, p, o))
    
# Add incoming triples taking this node as object, with 1 incoming hop
for s, p, o in g.triples((None, None, target_uri)):
    subgraph.add((s, p, o))

# Add outgoing triples taking this node as subject, with 2 outgoing hops
for s, p, o in g.triples((target_uri, None, None)):
    for s2, p2, o2 in g.triples((o, None, None)):
        subgraph.add((s2, p2, o2))

# Add incoming triples taking this node as object, with 2 incoming hops
for s, p, o in g.triples((None, None, target_uri)):
    for s2, p2, o2 in g.triples((s, None, None)):
        subgraph.add((s2, p2, o2))
        
print(f"Created subgraph with {len(subgraph)} triples")

Visualize the <code>subgraph</code> with <code>PyVis</code> as an interactive <code>HTML</code> object, saved to file. Depending on available compute power, the <code>subgraph</code> might be also displayed directly in the notebook.

In [ ]:
from pyvis.network import Network
from IPython.display import HTML, display

# Create PyVis network
net = Network(notebook=True, cdn_resources='inline')

# Add RDF subjects and objects as nodes, predicates as edges
for s, p, o in subgraph:
    net.add_node(str(s))
    net.add_node(str(o))
    net.add_edge(str(s), str(o), label=str(p))

# Display
net.show("subgraph.html")
display(HTML("subgraph.html"))

print(f"Subgraph with {len(subgraph)} triples saved as an html file")

<h4><CODE>graphviz</code> visualization a graph subset.</h4>

Create and visualize a different <code>subgraph2</code> with <code>graphviz</code> as a static graph, saved to file as <code>png</code>. Depending on available compute power, the <code>subgraph2</code> might be also displayed directly in the notebook.

In [ ]:
!pip install pydotplus

In [ ]:
!pip install graphviz

In [ ]:
# Source - https://stackoverflow.com/a/61483971
# Posted by Christian Wartena
License - CC BY-SA 4.0

import io
import pydotplus
from IPython.display import display, Image
from rdflib.tools.rdf2dot import rdf2dot
from rdflib import Graph, URIRef

g = Graph()
g.parse(output_file, format="turtle")

target_uri_1 = URIRef("https://a-warc-id-from-the-list-output-above.com")
target_uri_2 = URIRef("https://another-warc-id-from-the-list-output-above.com")

subgraph2 = Graph()
for target in [target_uri_1, target_uri_2]:
    for s, p, o in g.triples((target, None, None)):
        subgraph2.add((s, p, o))
    for s, p, o in g.triples((None, None, target)):
        subgraph2.add((s, p, o))

print(f"Created subgraph2 with {len(subgraph2)} triples")

def visualize(g, output_file="subgraph2.png"):
    stream = io.StringIO()
    rdf2dot(g, stream)
    dg = pydotplus.graph_from_dot_data(stream.getvalue())
    png = dg.create_png()
    display(Image(png))

    with open(output_file, "wb") as f:
        f.write(png)
     
visualize(subgraph2)

In [ ]:
Visualize <code>subgraph2</code> with <code>matplotlib.pyplo</code> as a static graph, saved to file as <code>png</code>. Depending on available compute power, the <code>subgraph2</code> might be also displayed directly in the notebook.

In [ ]:
# Source - https://stackoverflow.com/a/54095269
# Posted by Tom Hemmes, modified by community. See post 'Timeline' for change history
License - CC BY-SA 4.0

import rdflib
from rdflib.extras.external_graph_libs import rdflib_to_networkx_multidigraph
import networkx as nx
import matplotlib.pyplot as plt

G = rdflib_to_networkx_multidigraph(subgraph2)

# Plot Networkx instance of RDF Graph
pos = nx.spring_layout(G, scale=2)
edge_labels = nx.get_edge_attributes(G, 'r')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)
nx.draw(G, with_labels=True)

# Save it as *.png file
plt.savefig("subgraph2.png", dpi=300, bbox_inches="tight")

#if not in interactive mode for 
plt.show()